# 09 — Evaluation on Train/Validation: Selecting the Best Model × Prompt Combination

**Goal:** compare every (model, prompting strategy) combination produced by `run_extraction.py --split train_val` against the annotated ground-truth JSON, and select the winner to be applied — once — to the test split.

**Metrics** (entity level, micro-averaged across the 16 notes):

| Criterion | Definition |
|---|---|
| **Exact** P / R / F1 | predicted span boundaries identical to the gold span |
| **Relaxed** P / R / F1 | each boundary may deviate by ≤ 10% of the gold span length |

**Selection criterion:** relaxed F1 (primary), with exact F1 as tie-breaker. Relaxed F1 is primary because minor boundary differences (e.g. trailing punctuation) rarely matter clinically; exact match is reported to quantify how often the model nails the boundaries perfectly.

**Caveat for the thesis:** with 16 notes, small metric differences between combinations may be noise — report the full table and comment on how close the runner-ups are, not just the winner.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from clinical_notes_extraction.utils.evaluation import evaluate_split, locate_spans

In [ ]:
# Constants local to this notebook
RESULTS_DIR = Path("results/train_val")
GROUND_TRUTH_FILE = Path("data/annotations/ground_truth.json")
SPLIT_FILE = Path("data/splits/train_val.csv")
METRICS_FILE = Path("results/train_val_metrics.csv")
BEST_COMBO_FILE = Path("results/best_combination.json")

## Load ground truth and localise gold spans

The ground-truth JSON maps `note_id -> [{"span": "verbatim text", ...}, ...]`. If the annotations do not carry `start`/`end` offsets, they are localised here with the same verbatim-search logic used for the model outputs — so both sides of the comparison are grounded identically.

In [ ]:
with open(GROUND_TRUTH_FILE) as f:
    ground_truth_raw = json.load(f)

notes = pd.read_csv(SPLIT_FILE).set_index("note_id")

ground_truth = {}
for note_id, entities in ground_truth_raw.items():
    if note_id not in notes.index.astype(str).tolist() and int(note_id) not in notes.index:
        continue  # this notebook only evaluates the train/val notes
    note_text = notes.loc[int(note_id) if note_id.isdigit() else note_id, "text"]
    if entities and "start" not in entities[0]:
        entities = locate_spans(note_text, [e["span"] for e in entities])
        unverified = [e["span"] for e in entities if not e["verified"]]
        assert not unverified, f"Gold spans not found verbatim in note {note_id}: {unverified}"
    ground_truth[str(note_id)] = entities

print(f"Ground truth loaded for {len(ground_truth)} train/val notes")

## Load predictions

One file per (model, strategy, note): `results/train_val/<model>/<strategy>/<note_id>.json`.

In [ ]:
predictions = {}  # (model, strategy) -> {note_id: [entities]}

for model_dir in sorted(p for p in RESULTS_DIR.iterdir() if p.is_dir()):
    for strategy_dir in sorted(p for p in model_dir.iterdir() if p.is_dir()):
        combo = (model_dir.name, strategy_dir.name)
        predictions[combo] = {}
        for pred_file in sorted(strategy_dir.glob("*.json")):
            with open(pred_file) as f:
                pred = json.load(f)
            predictions[combo][str(pred["note_id"])] = pred["medications"]

print(f"Loaded predictions for {len(predictions)} (model, strategy) combinations")

## Compute metrics per combination

In [ ]:
rows = []
for (model, strategy), preds in predictions.items():
    metrics = evaluate_split(preds, ground_truth)
    rows.append({"model": model, "strategy": strategy, **metrics})

metrics_df = (
    pd.DataFrame(rows)
    .sort_values(["relaxed_f1", "exact_f1"], ascending=False)
    .reset_index(drop=True)
)

METRICS_FILE.parent.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(METRICS_FILE, index=False)
print(f"Saved: {METRICS_FILE}\n")

display_cols = ["model", "strategy",
                "exact_precision", "exact_recall", "exact_f1",
                "relaxed_precision", "relaxed_recall", "relaxed_f1"]
metrics_df[display_cols].round(3)

## Visual comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, metric in zip(axes, ["exact_f1", "relaxed_f1"]):
    pivot = metrics_df.pivot(index="model", columns="strategy", values=metric)
    im = ax.imshow(pivot.values, cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(pivot.columns)), pivot.columns)
    ax.set_yticks(range(len(pivot.index)), pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f"{pivot.values[i, j]:.2f}", ha="center", va="center", color="white")
    ax.set_title(metric.replace("_", " ").title() + " (train/val)")
    fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

## Select and freeze the winning combination

The winner is persisted so the test phase reads it programmatically — no manual copy-paste between notebooks and scripts.

In [ ]:
best = metrics_df.iloc[0]
best_combo = {
    "model": best["model"].replace("_", ":"),
    "strategy": best["strategy"],
    "train_val_relaxed_f1": round(best["relaxed_f1"], 4),
    "train_val_exact_f1": round(best["exact_f1"], 4),
}

with open(BEST_COMBO_FILE, "w") as f:
    json.dump(best_combo, f, indent=2)

print("Best combination:", json.dumps(best_combo, indent=2))
print("\nRun the final test phase (once) with:")
print(f"  python scripts/run_extraction.py --split test "
      f"--models {best_combo['model']} --strategies {best_combo['strategy']}")